In [ ]:
import os
import sys
import math
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple

##############################################################################
# 1. SAFE OPERATIONS
##############################################################################
def safe_div(a: float, b: float) -> float:
    """
    Safely divides a by b. If |b| < 1e-12, returns 1.0 to avoid
    division-by-zero or huge magnitudes.
    """
    if abs(b) < 1e-12:
        return 1.0
    return a / b

def safe_log(a: float) -> float:
    """
    Safely computes log(|a|). Returns 0.0 if |a| < 1e-12
    to avoid log(0) or negative domain.
    """
    if abs(a) < 1e-12:
        return 0.0
    return math.log(abs(a))

def safe_exp(a: float) -> float:
    """
    Safely computes exp(a). If |a| is too large, clamp to [-50, 50].
    """
    MAX_EXP = 50
    MIN_EXP = -50
    if a > MAX_EXP:
        return math.exp(MAX_EXP)
    elif a < MIN_EXP:
        return math.exp(MIN_EXP)
    else:
        return math.exp(a)

def safe_sin(a: float) -> float:
    """Sine function (won't overflow)."""
    return math.sin(a)

def safe_cos(a: float) -> float:
    """Cosine function (won't overflow)."""
    return math.cos(a)

##############################################################################
# 2. FUNCTION MAP (Symbol -> (callable, arity))
##############################################################################
FUNCTION_MAP = {
    '+':   (lambda x, y: x + y, 2),
    '-':   (lambda x, y: x - y, 2),
    '*':   (lambda x, y: x * y, 2),
    '/':   (safe_div,            2),
    'sin': (safe_sin,            1),
    'cos': (safe_cos,            1),
    'exp': (safe_exp,            1),
    'log': (safe_log,            1),
}

# Terminal set: single variable 'x' plus constants as strings
TERMINALS = ['x']
CONSTANTS = [str(c) for c in range(1, 6)]  # '1','2','3','4','5'

##############################################################################
# 3. CHROMOSOME CLASS
##############################################################################
class Chromosome:
    """
    A Chromosome consists of 'num_genes' genes, each gene having:
      - head (can contain functions or terminals)
      - tail (can contain only terminals or constants)

    We'll store the entire chromosome as a list of symbols.
    """

    def __init__(self, head_length: int, tail_length: int, num_genes: int = 1):
        self.head_length = head_length
        self.tail_length = tail_length
        self.num_genes   = num_genes
        self.chromosome  = self._create_random_chromosome()
        self.fitness     = None  # assigned later

    def _create_random_chromosome(self) -> List[str]:
        """
        Builds a new random chromosome with 'num_genes' genes,
        each having (head_length + tail_length) symbols.
        """
        new_chrom = []
        for _ in range(self.num_genes):
            # Head => can be function or terminal
            head = [
                random.choice(list(FUNCTION_MAP.keys()) + TERMINALS + CONSTANTS)
                for _ in range(self.head_length)
            ]
            # Tail => terminal or constant only
            tail = [
                random.choice(TERMINALS + CONSTANTS)
                for _ in range(self.tail_length)
            ]
            new_chrom.extend(head + tail)
        return new_chrom

##############################################################################
# 4. EVALUATION: STACK-BASED
##############################################################################
def evaluate_gene(gene: List[str], x_val: float) -> float:
    """
    Evaluates a single gene (list of symbols) using Karva (right-to-left)
    postfix notation. We only call safe_* functions to avoid OverflowError.
    """
    stack = []
    for symbol in reversed(gene):
        if symbol in FUNCTION_MAP:
            func, arity = FUNCTION_MAP[symbol]
            if len(stack) < arity:
                return 0.0  # invalid gene => 0
            if arity == 2:
                a = stack.pop()
                b = stack.pop()
                stack.append(func(a, b))
            else:
                a = stack.pop()
                stack.append(func(a))
        elif symbol == 'x':
            stack.append(x_val)
        else:
            # interpret as float constant
            try:
                cval = float(symbol)
            except ValueError:
                cval = 0.0
            stack.append(cval)

    return stack[0] if len(stack) == 1 else 0.0

def evaluate_chromosome(chrom: Chromosome, x_val: float) -> float:
    """
    Evaluate an entire chromosome by summing each gene's output for a given x_val.
    """
    gene_len = chrom.head_length + chrom.tail_length
    total_output = 0.0
    for g in range(chrom.num_genes):
        start = g * gene_len
        end   = start + gene_len
        total_output += evaluate_gene(chrom.chromosome[start:end], x_val)
    return total_output

def compute_fitness(chrom: Chromosome, X: np.ndarray, Y: np.ndarray) -> float:
    """
    Negative MSE => higher fitness = lower MSE.
    """
    predictions = [evaluate_chromosome(chrom, x) for x in X]
    mse = np.mean((Y - np.array(predictions))**2)
    return -mse

##############################################################################
# 5. TRANSLATING A CHROMOSOME INTO A READABLE MATH EXPRESSION
##############################################################################
def gene_to_infix(gene: List[str]) -> str:
    """
    Converts a gene (list of symbols) into an infix math expression (string).
    We'll do a stack-based parse similar to evaluate_gene, 
    but instead of computing numeric values, we build string expressions.
    """
    stack = []
    for symbol in reversed(gene):
        if symbol in FUNCTION_MAP:
            # Determine how many operands are needed
            _, arity = FUNCTION_MAP[symbol]
            if len(stack) < arity:
                return "0"  # invalid
            if arity == 2:
                a = stack.pop()
                b = stack.pop()
                # We'll create an infix string: "(a symbol b)"
                # e.g., "(x + 3.0)" or "( (x) / (7.0) )"
                expr = f"({a} {symbol} {b})"
            else:
                # unary operator, e.g. sin(a)
                a = stack.pop()
                expr = f"{symbol}({a})"
            stack.append(expr)
        else:
            # 'x' or constant
            stack.append(symbol)
    return stack[0] if len(stack) == 1 else "0"

def chromosome_to_expression(chrom: Chromosome) -> str:
    """
    Converts each gene in the chromosome into an infix expression
    and sums them with '+' (the simplest linking function).
    """
    gene_len = chrom.head_length + chrom.tail_length
    expressions = []
    for g in range(chrom.num_genes):
        start = g * gene_len
        end   = start + gene_len
        gene_expr = gene_to_infix(chrom.chromosome[start:end])
        expressions.append(gene_expr)
    # Link by +. If there's only 1 gene, it's just that gene's expression.
    return " + ".join(expressions)

##############################################################################
# 6. SELECTION: ROULETTE WHEEL
##############################################################################
def roulette_wheel_selection(population: List[Chromosome]) -> Chromosome:
    """
    Select from population with probability ~ (shifted) fitness.
    """
    valid = [c for c in population if c.fitness is not None and c.fitness != -float('inf')]
    if not valid:
        return random.choice(population)

    min_fit = min(ch.fitness for ch in valid)
    offset = 0.0
    if min_fit < 0:
        offset = -min_fit + 1e-3

    total = 0.0
    partial_sums = []
    for ch in valid:
        total += (ch.fitness + offset)
        partial_sums.append(total)

    pick = random.uniform(0, total)
    for ch, cum_sum in zip(valid, partial_sums):
        if pick <= cum_sum:
            return ch
    return valid[-1]

##############################################################################
# 7. GENETIC OPERATORS
##############################################################################
def mutate(chrom: Chromosome, mutation_rate: float) -> Chromosome:
    """
    With mutation_rate probability, replace symbol with random valid symbol.
    """
    child = copy.deepcopy(chrom)
    g_len = child.head_length + child.tail_length

    for i in range(len(child.chromosome)):
        if random.random() < mutation_rate:
            # if we're in head => can be function or terminal
            if (i % g_len) < child.head_length:
                child.chromosome[i] = random.choice(list(FUNCTION_MAP.keys()) + TERMINALS + CONSTANTS)
            else:
                # tail => only terminal or constant
                child.chromosome[i] = random.choice(TERMINALS + CONSTANTS)
    return child

def transposition(chrom: Chromosome, transposition_rate: float) -> Chromosome:
    """
    Swaps two symbols within the same gene with probability transposition_rate.
    """
    if random.random() > transposition_rate:
        return chrom

    child = copy.deepcopy(chrom)
    g_len = child.head_length + child.tail_length

    # pick which gene
    gene_idx = random.randint(0, child.num_genes - 1)
    start = gene_idx * g_len
    end   = start + g_len

    pos1 = random.randint(start, end - 1)
    pos2 = random.randint(start, end - 1)
    child.chromosome[pos1], child.chromosome[pos2] = (
        child.chromosome[pos2],
        child.chromosome[pos1]
    )
    return child

def crossover(parent1: Chromosome, parent2: Chromosome) -> Tuple[Chromosome, Chromosome]:
    """
    Single-point crossover for each gene (assuming same structure).
    """
    if (parent1.num_genes != parent2.num_genes or
        parent1.head_length != parent2.head_length or
        parent1.tail_length != parent2.tail_length):
        return copy.deepcopy(parent1), copy.deepcopy(parent2)

    c1 = copy.deepcopy(parent1)
    c2 = copy.deepcopy(parent2)
    g_len = parent1.head_length + parent1.tail_length

    for g in range(parent1.num_genes):
        start = g * g_len
        end   = start + g_len
        if g_len <= 1:
            continue
        cp = random.randint(1, g_len - 1)
        c1.chromosome[start:start+cp], c2.chromosome[start:start+cp] = (
            c2.chromosome[start:start+cp],
            c1.chromosome[start:start+cp]
        )
    return c1, c2

##############################################################################
# 8. MAIN GEP LOOP
##############################################################################
def gep(
    training_data: pd.DataFrame,
    population_size: int = 60,
    generations: int = 300,
    head_length: int = 5,
    tail_length: int = 3,
    num_genes: int = 1,
    mutation_rate: float = 0.1,
    transposition_rate: float = 0.05,
    elitism: float = 0.1
) -> Tuple[Chromosome, List[float], List[float]]:
    """
    Core GEP routine:
      1) Initialize population
      2) For each generation: evaluate fitness, do selection, crossover, mutate
      3) Return best Chromosome + fitness history
    """
    if 'x' not in training_data.columns or 'y' not in training_data.columns:
        raise ValueError("training_data must have columns 'x' and 'y'.")

    X = training_data['x'].values
    Y = training_data['y'].values

    # 1) Initialize population
    population = [
        Chromosome(head_length, tail_length, num_genes=num_genes)
        for _ in range(population_size)
    ]

    best_fit_hist = []
    avg_fit_hist  = []

    for gen in range(generations):
        # 2) Evaluate
        for chrom in population:
            chrom.fitness = compute_fitness(chrom, X, Y)

        # Sort descending by fitness
        population.sort(key=lambda c: c.fitness, reverse=True)
        best_fit = population[0].fitness

        # Average fit among valid
        valid_fits = [c.fitness for c in population if c.fitness != -float('inf')]
        avg_fit = sum(valid_fits)/len(valid_fits) if valid_fits else -float('inf')

        best_fit_hist.append(best_fit)
        avg_fit_hist.append(avg_fit)

        # Print progress (convert from fitness to MSE by negating)
        best_mse = -best_fit
        if avg_fit != -float('inf'):
            avg_mse = -avg_fit
            print(f"Gen {gen+1}/{generations}: Best MSE={best_mse:.4f}, Avg MSE={avg_mse:.4f}")
        else:
            print(f"Gen {gen+1}/{generations}: All solutions invalid.")

        # 3) Next generation
        new_pop = []
        elite_num = max(1, int(elitism * population_size))
        new_pop.extend(population[:elite_num])  # Elitism

        while len(new_pop) < population_size:
            p1 = roulette_wheel_selection(population)
            p2 = roulette_wheel_selection(population)
            c1, c2 = crossover(p1, p2)
            c1 = mutate(c1, mutation_rate)
            c2 = mutate(c2, mutation_rate)
            c1 = transposition(c1, transposition_rate)
            c2 = transposition(c2, transposition_rate)
            new_pop.extend([c1, c2])

        population = new_pop[:population_size]

    # Final best
    for chrom in population:
        chrom.fitness = compute_fitness(chrom, X, Y)
    population.sort(key=lambda c: c.fitness, reverse=True)
    best_chrom = population[0]
    return best_chrom, best_fit_hist, avg_fit_hist

##############################################################################
# 9. APPLY & VISUALIZE
##############################################################################
def apply_chromosome(chrom: Chromosome, X_vals: np.ndarray) -> np.ndarray:
    """Evaluate the best chromosome on a vector of X_vals."""
    return np.array([evaluate_chromosome(chrom, x) for x in X_vals])

def visualize_results(
    training_data: pd.DataFrame,
    best_chromosome: Chromosome,
    best_fit_hist: List[float],
    avg_fit_hist: List[float],
    testing_data: pd.DataFrame = None
):
    """
    Create a 3-panel figure:
      (1) Training data actual vs predicted
      (2) Fitness progress (MSE) 
      (3) Test data predicted vs actual (if 'y' present)
    """
    plt.style.use('dark_background')
    fig = plt.figure(figsize=(18, 5))
    fig.suptitle("GEP Results (Overflow-Protected) + Equation Printing", fontsize=16, color='w')

    # 1) Training
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.scatter(training_data['x'], training_data['y'], color='cyan', label='Train Actual')
    y_pred_train = apply_chromosome(best_chromosome, training_data['x'].values)
    ax1.scatter(training_data['x'], y_pred_train, color='magenta', label='Train Predicted')
    ax1.set_title("Training Data", color='w')
    ax1.set_xlabel("x", color='w')
    ax1.set_ylabel("y", color='w')
    ax1.legend(facecolor='gray')

    # 2) Fitness Progress
    ax2 = fig.add_subplot(1, 3, 2)
    best_mse = [-bf for bf in best_fit_hist]
    avg_mse  = [-af for af in avg_fit_hist]
    ax2.plot(range(len(best_mse)), best_mse, color='lime', label='Best MSE')
    ax2.plot(range(len(avg_mse)), avg_mse, color='yellow', linestyle='--', label='Avg MSE')
    ax2.set_title("Fitness Progress (MSE)", color='w')
    ax2.set_xlabel("Generation", color='w')
    ax2.set_ylabel("MSE", color='w')
    ax2.legend(facecolor='gray')

    # 3) Testing Data
    ax3 = fig.add_subplot(1, 3, 3)
    if testing_data is not None and 'x' in testing_data.columns:
        X_test = testing_data['x'].values
        y_pred_test = apply_chromosome(best_chromosome, X_test)
        ax3.scatter(X_test, y_pred_test, color='orange', label='Test Predicted')
        if 'y' in testing_data.columns:
            ax3.scatter(X_test, testing_data['y'], color='green', label='Test Actual')
        ax3.set_title("Testing Data", color='w')
        ax3.set_xlabel("x", color='w')
        ax3.set_ylabel("y", color='w')
        ax3.legend(facecolor='gray')
    else:
        ax3.set_visible(False)

    plt.tight_layout()
    plt.show()

##############################################################################
# 10. MAIN DEMO
##############################################################################
def main():
    """
    1) Reads data/training_data.csv and data/testing_data.csv
    2) Runs GEP
    3) Prints final best MSE and the discovered equation
    4) Visualizes results
    """
    train_path = os.path.join('training_data.csv')
    test_path  = os.path.join('test_data.csv')

    # Load training data
    try:
        training_data = pd.read_csv(train_path)
    except Exception as e:
        print(f"Error reading {train_path}: {e}")
        sys.exit(1)

    # Attempt to load test data
    testing_data = None
    if os.path.exists(test_path):
        try:
            testing_data = pd.read_csv(test_path)
        except Exception as e:
            print(f"Error reading {test_path}: {e}")

    # Run GEP
    best_chrom, best_fit_hist, avg_fit_hist = gep(
        training_data=training_data,
        population_size=100,
        generations=1000,
        head_length=7,
        tail_length=5,
        num_genes=1,
        mutation_rate=0.18,
        transposition_rate=0.8,
        elitism=0.
    )

    # Compute final best MSE
    final_mse = -best_chrom.fitness if best_chrom.fitness is not None else 9999
    print(f"\nFinal Best MSE: {final_mse:.4f}")

    # Translate best chromosome to a symbolic expression
    equation_str = chromosome_to_expression(best_chrom)
    print("\nDiscovered Equation:")
    print(equation_str)

    # Visualize
    visualize_results(training_data, best_chrom, best_fit_hist, avg_fit_hist, testing_data)

if __name__ == "__main__":
    main()


In [2]:
testing_data

NameError: name 'testing_data' is not defined

In [ ]:
y = np.sin(x**2) * np.cos(x / 2) + (x**3 / 10) - np.exp(-np.abs(x) / 2)